# 1. Data preprocessing

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# Load the edge data
edges_data = pd.read_csv('data/large_twitch_edges.csv')

# Load the feature data
features_data = pd.read_csv('data/large_twitch_features.csv')

In [ ]:
# Create a directed graph
G = nx.DiGraph()

# Ensure required columns are in the dataframe
if 'numeric_id_1' in edges_data.columns and 'numeric_id_2' in edges_data.columns:
    # Add edges to the graph
    for _, row in edges_data.iterrows():
        G.add_edge(row['numeric_id_1'], row['numeric_id_2'])
else:
    print("Required columns are not in the edges_data")


# Ensure required columns are in the dataframe
required_columns = ['numeric_id', 'views', 'life_time', 'created_at', 'updated_at', 'language']
if all(col in features_data.columns for col in required_columns):
    # Add node attributes to the graph
    for _, row in features_data.iterrows():
        node_id = row['numeric_id']
        G.nodes[node_id]['views'] = row['views']
        G.nodes[node_id]['life_time'] = row['life_time']
        G.nodes[node_id]['created_at'] = row['created_at']
        G.nodes[node_id]['updated_at'] = row['updated_at']
        G.nodes[node_id]['language'] = row['language']
else:
    print("Required columns are not in the features_data")

In [ ]:
import random

# Random sampling of nodes
sample_size = 10000
sampled_nodes = random.sample(G.nodes(), sample_size)
G_sampled = G.subgraph(sampled_nodes)

# Focus on the largest connected component of the sampled graph
largest_cc = max(nx.weakly_connected_components(G_sampled), key=len)
G_final = G_sampled.subgraph(largest_cc)

# Check the size of the final graph
print(f"Final graph has {G_final.number_of_nodes()} nodes and {G_final.number_of_edges()} edges")



In [ ]:
# Visualize the sampled graph
plt.figure(figsize=(12, 12))
pos = nx.spring_layout(G_final)
nx.draw(G_final, pos, node_color='skyblue', node_size=50, edge_color='gray', with_labels=False)
plt.title('Sampled Twitch Social Network')
plt.show()

# Save the sampled graph for further analysis
nx.write_edgelist(G_final, 'sampled_graph.edgelist')